## Эксперименты с Qwen2.5-0.5B и разными оптимизаторами

Этот ноутбук автоматически:
1. Настраивает окружение (Colab / Kaggle / локально)
2. Клонирует репозиторий `huawei-research`
3. Устанавливает зависимости
4. Подключает ClearML для отслеживания экспериментов
5. Авторизует Hugging Face Hub (опционально)

**Перед запуском** убедитесь, что выбран GPU:  
- Colab: `Среда выполнения → Сменить тип среды выполнения → T4`  
- Kaggle: `Settings → Accelerator → GPU`

## 1. Определение среды и рабочей директории

In [ ]:
import os
import sys
from pathlib import Path

# Определяем среду выполнения
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules

if IN_COLAB:
    WORK_DIR = "/content"
    MOUNT_DRIVE = True
    print("✅ Среда: Google Colab")
elif IN_KAGGLE:
    WORK_DIR = "/kaggle/working"
    MOUNT_DRIVE = False
    print("✅ Среда: Kaggle")
else:
    WORK_DIR = Path.cwd()
    MOUNT_DRIVE = False
    print("✅ Среда: локальная")

PROJECT_DIR = Path(WORK_DIR) / "huawei-research"
OUTPUT_BASE = PROJECT_DIR / "outputs"

print(f"📁 Рабочая директория: {WORK_DIR}")
print(f"📁 Проект будет склонирован в: {PROJECT_DIR}")

## 2. Монтирование Google Drive (только для Colab)

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
        print("✅ Google Drive примонтирован к /content/drive")
    except Exception as e:
        print(f"⚠️ Не удалось примонтировать Drive: {e}")
        print("   Продолжаем без монтирования (модели будут сохраняться локально).")
else:
    print("⏩ Монтирование Drive не требуется в этой среде.")

## 3. Клонирование репозитория

In [ ]:
if not PROJECT_DIR.exists():
    print(f"📦 Клонируем репозиторий в {PROJECT_DIR} ...")
    !git clone https://github.com/Se1ecta/huawei-research.git {PROJECT_DIR}
else:
    print(f"✅ Репозиторий уже существует: {PROJECT_DIR}")
    # Опционально: обновить
    # !git -C {PROJECT_DIR} pull

## 4. Установка зависимостей

In [ ]:
# Устанавливаем необходимые пакеты
%pip install -q clearml transformers==5.5.4 accelerate datasets

## 5. Настройка ClearML

Вам понадобятся **API access key** и **secret key** из вашего аккаунта ClearML (можно получить на [app.clear.ml](https://app.clear.ml) → Settings → Workspace → Create new credentials).


In [ ]:
import os
from google.colab import userdata

# Пытаемся получить ключи из Colab Secrets
try:
    access_key = userdata.get("CLEARML_API_ACCESS_KEY")
    secret_key = userdata.get("CLEARML_API_SECRET_KEY")
    print("✅ Ключи ClearML загружены из Colab Secrets")
except Exception as e:
    print("⚠️ Не найдены секреты в Colab. Введите ключи вручную (небезопасно для шаринга).")
    access_key = input("Введите CLEARML_API_ACCESS_KEY: ")
    secret_key = input("Введите CLEARML_API_SECRET_KEY: ")

# Устанавливаем переменные окружения для ClearML
os.environ["CLEARML_WEB_HOST"] = "https://app.clear.ml/"
os.environ["CLEARML_API_HOST"] = "https://api.clear.ml"
os.environ["CLEARML_FILES_HOST"] = "https://files.clear.ml"
os.environ["CLEARML_API_ACCESS_KEY"] = access_key
os.environ["CLEARML_API_SECRET_KEY"] = secret_key
os.environ["LOGLEVEL"] = "INFO"

print("✅ ClearML настроен. При запуске обучения эксперимент автоматически появится в веб-интерфейсе.")

## 6. Авторизация Hugging Face Hub

Чтобы пушить модели на Hub (опция `--push_to_hub=True`, если не нужно выставить на False), нужен токен доступа. Получить его можно на [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) (тип `write`).

In [ ]:
from huggingface_hub import notebook_login

notebook_login()
# Появится виджет для ввода токена. Вставьте токен и нажмите "Login".

## 7. Проверка GPU и системной информации

In [ ]:
import torch

# Проверяем, доступен ли GPU
if torch.cuda.is_available():
    print(f"✅ GPU найден: {torch.cuda.get_device_name(0)}")
    print(f"   Количество памяти: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ GPU не обнаружен. Обучение будет очень медленным. Перезапустите среду выполнения и выберите GPU.")
    
# Проверка наличия train.py
train_script = PROJECT_DIR / 'src' / 'train.py'
if not train_script.exists():
    raise FileNotFoundError(f"❌ Не найден {train_script}. Убедитесь, что репозиторий склонирован корректно.")
else:
    print(f"✅ Скрипт обучения найден: {train_script}")

## 8. Функция запуска эксперимента

Команда ниже запускает обучение модели **Qwen/Qwen2.5-0.5B** на 1 эпоху с оптимизатором AdamW.
Все метрики и логи автоматически отправятся в ClearML.

In [ ]:
run_exp_cmd = f"""python {train_script} \
  --model_name Qwen/Qwen2.5-0.5B \
  --optimizer adamw \
  --zo_eps 1e-3 \
  --per_device_train_batch_size 2 \
  --gradient_accumulation_steps 8 \
  --learning_rate 2e-5 \
  --seq_length 512 \
  --num_train_epochs 1 \
  --lr_scheduler_type cosine \
  --warmup_steps 0.01 \
  --weight_decay 0 \
  --logging_steps 10 \
  --logging_strategy="steps" \
  --push_to_hub True \
  --report_to clearml \
  --seed 42 \
  --output_dir ./Qwen2.5-0.5B_adamw"""

!{run_exp_cmd} 

## 9. Установка Evaluation Harness из GitHub

In [ ]:
# Клонируем репозиторий lm-evaluation-harness
LM_EVAL_DIR = Path(WORK_DIR) / "lm-evaluation-harness"

if not LM_EVAL_DIR.exists():
    print("📦 Клонируем lm-evaluation-harness...")
    !git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness {LM_EVAL_DIR}
else:
    print("✅ Репозиторий уже существует, обновляем...")
    !git -C {LM_EVAL_DIR} pull

# Устанавливаем в режиме editable
print("⚙️ Устанавливаем lm_eval...")
%pip install -e {LM_EVAL_DIR}

print("✅ lm_eval успешно установлен из GitHub.")

## 10. Оценка

In [ ]:
TASKS = "piqa,arc_easy,arc_challenge,winogrande,hellaswag"
BATCH_SIZE = 8
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
MODEL_PATH="Qwen/Qwen2.5-0.5B"
OUTPUT_RESULT_PATH="./results"
cmd = f"""
    lm_eval --model hf \
        --model_args pretrained={MODEL_PATH},trust_remote_code=True \
        --tasks {TASKS} \
        --device {DEVICE} \
        --batch_size {BATCH_SIZE} \
        --output_path {OUTPUT_RESULT_PATH} \
        --trust_remote_code
    """
    # Выполняем
!{cmd}